# [3] 투표소 정보 수집 (사전투표소 + 선거일투표소)

**End Point**: `https://apis.data.go.kr/9760000/PolplcInfoInqireService2`

| 오퍼레이션 | 설명 |
|---|---|
| `getPrePolplcOtlnmapTrnsportInfoInqire` | 사전투표소 |
| `getPolplcOtlnmapTrnsportInfoInqire` | 선거일투표소 |

## 응답 컬럼 (PDF 확인)

| 컬럼명 | 사전투표소 | 선거일투표소 |
|---|---|---|
| evPsName | 사전투표소명 | - |
| psName | - | 선거일투표소명 |
| sdName | 시도명 | 시도명 |
| wiwName | 구시군명 | 구시군명 |
| emdName | 읍면동명 | 읍면동명 |
| evOrder | 순서 | - |
| placeName | 건물명 | 장소명 |
| addr | 주소 | 주소 |
| floor | 층 | 층 |

> ⚠️ **선거 기간 중에만 제공** — 종료된 선거는 빈 응답이 반환됩니다.  
> `sdName`이 필수이므로 17개 시도를 순회하며 수집합니다.

In [10]:
# ✏️ 본인 Decoding 키로 교체하세요
API_KEY = "6lVhhlLRaGq/+tidZgS0POWCcl7BOqRJXiDj+Xtl/+rJJVEqNPWFjwFpyWkOn3NaNqacOHvj9UG+BnHAGBFd4w=="

In [11]:
import requests
import pandas as pd
import time

def fetch_all_pages(base_url, fixed_params, page_size=100, delay=0.3):
    all_items = []
    page = 1
    while True:
        params = {**fixed_params, "pageNo": str(page), "numOfRows": str(page_size), "resultType": "json"}
        try:
            resp = requests.get(base_url, params=params, timeout=15)
            if resp.status_code != 200:
                print(f"  ⚠️  HTTP {resp.status_code}: {resp.text[:300]}")
                break
            data = resp.json()
            header = data["response"]["header"]
            if header["resultCode"] not in ("INFO-00", "00"):
                print(f"  ⚠️  API 오류: {header['resultMsg']}")
                break
            body  = data["response"]["body"]
            total = int(body.get("totalCount", 0))
            raw   = body.get("items") or {}
            items = raw.get("item", []) if isinstance(raw, dict) else []
            if isinstance(items, dict):
                items = [items]
            all_items.extend(items)
            print(f"  페이지 {page}: {len(items)}건  (누적 {len(all_items)}/{total})")
            if len(all_items) >= total or not items:
                break
            page += 1
            time.sleep(delay)
        except Exception as e:
            print(f"  ❌ 오류 (페이지 {page}): {e}")
            break
    return all_items

def save_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n💾 저장 완료: {path}  ({len(df)}행 × {len(df.columns)}열)")

In [12]:
ELECTIONS = {
    "20200415": "제21대 국회의원선거",
    "20220309": "제20대 대통령선거",
    "20220601": "제8회 전국동시지방선거",
    "20231011": "제22대 국선 강서구청장 보궐선거",
    "20240410": "제22대 국회의원선거",
    "20250603": "제21대 대통령선거",
}

In [13]:
BASE = "https://apis.data.go.kr/9760000/PolplcInfoInqireService2"

ENDPOINTS = {
    "사전투표소":   f"{BASE}/getPrePolplcOtlnmapTrnsportInfoInqire",
    "선거일투표소": f"{BASE}/getPolplcOtlnmapTrnsportInfoInqire",
}

SD_NAMES = [
    "서울특별시", "부산광역시", "대구광역시", "인천광역시", "광주광역시",
    "대전광역시", "울산광역시", "세종특별자치시", "경기도", "강원특별자치도",
    "충청북도", "충청남도", "전북특별자치도", "전라남도", "경상북도",
    "경상남도", "제주특별자치도"
]

all_records = []
for poll_type, url in ENDPOINTS.items():
    print(f"\n{'#'*55}")
    print(f"🗳️  {poll_type}")
    print(f"{'#'*55}")
    for sg_id, election_name in ELECTIONS.items():
        print(f"\n  📌 {election_name}  (sgId={sg_id})")
        for sd in SD_NAMES:
            items = fetch_all_pages(
                base_url=url,
                fixed_params={"serviceKey": API_KEY, "sgId": sg_id, "sdName": sd},
            )
            for item in items:
                item["election_name"] = election_name
                item["polling_type"]  = poll_type
            all_records.extend(items)
            if items:
                print(f"     {sd}: {len(items)}건")
        time.sleep(0.3)

print(f"\n총 {len(all_records)}건 수집 완료")


#######################################################
🗳️  사전투표소
#######################################################

  📌 제21대 국회의원선거  (sgId=20200415)
  페이지 1: 100건  (누적 100/424)
  페이지 2: 100건  (누적 200/424)
  페이지 3: 100건  (누적 300/424)
  페이지 4: 100건  (누적 400/424)
  페이지 5: 24건  (누적 424/424)
     서울특별시: 424건
  페이지 1: 100건  (누적 100/205)
  페이지 2: 100건  (누적 200/205)
  페이지 3: 5건  (누적 205/205)
     부산광역시: 205건
  페이지 1: 100건  (누적 100/139)
  페이지 2: 39건  (누적 139/139)
     대구광역시: 139건
  페이지 1: 100건  (누적 100/157)
  페이지 2: 57건  (누적 157/157)
     인천광역시: 157건
  페이지 1: 95건  (누적 95/95)
     광주광역시: 95건
  페이지 1: 80건  (누적 80/80)
     대전광역시: 80건
  페이지 1: 56건  (누적 56/56)
     울산광역시: 56건
  페이지 1: 19건  (누적 19/19)
     세종특별자치시: 19건
  페이지 1: 100건  (누적 100/544)
  페이지 2: 100건  (누적 200/544)
  페이지 3: 100건  (누적 300/544)
  페이지 4: 100건  (누적 400/544)
  페이지 5: 100건  (누적 500/544)
  페이지 6: 44건  (누적 544/544)
     경기도: 544건
  ⚠️  API 오류: 데이터 정보가 없습니다. 입력 파라미터값을 확인해주시기 바랍니다.
  페이지 1: 100건  (누적 100/154)
  페이지 2: 54건  (누적

In [14]:
if not all_records:
    print("⚠️  수집된 데이터 없음 (선거 기간 외 미제공)")
else:
    df = pd.DataFrame(all_records)

    # PDF 확인된 실제 컬럼명
    col_map = {
        "election_name": "선거명",
        "polling_type":  "투표소구분",
        "sgId":          "선거ID",
        "sdName":        "시도명",
        "wiwName":       "구시군명",
        "emdName":       "읍면동명",
        "evPsName":      "사전투표소명",   # 사전투표소
        "psName":        "선거일투표소명", # 선거일투표소
        "evOrder":       "순서",
        "placeName":     "건물명(장소명)",
        "addr":          "주소",
        "floor":         "층",
        "num":           "결과순서",
    }
    existing = [c for c in col_map if c in df.columns]
    df = df[existing].rename(columns=col_map)

    display(df.head(10))
    save_csv(df, "03_투표소정보.csv")

,선거명,투표소구분,선거ID,시도명,구시군명,읍면동명,사전투표소명,선거일투표소명,순서,건물명(장소명),주소,층,결과순서
0,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,삼청동,삼청동사전투표소,NaN,1,"삼청동주민센터(1층, 민원실)",서울특별시 종로구 삼청로 107 (삼청동),1층,1
1,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,부암동,부암동사전투표소,NaN,1,"석파랑 신관(지하3층, 갤러리)",서울특별시 종로구 자하문로 307 (홍지동),지하3층,2
2,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,평창동,평창동사전투표소,NaN,1,"평창동주민센터(4층, 대강당)",서울특별시 종로구 평창문화로 65 (평창동),4층,3
3,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,무악동,무악동사전투표소,NaN,1,"무악동주민센터(4층, 대강당)",서울특별시 종로구 통일로14길 36 (무악동),4층,4
4,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,교남동,교남동사전투표소,NaN,1,"교남동주민센터(4층, 강당)",서울특별시 종로구 송월길 154 (행촌동),4층,5
5,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,가회동,가회동사전투표소,NaN,1,"가회동주민센터(지하1층, 대강당)",서울특별시 종로구 북촌로 35 (가회동),지하1층,6
6,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,종로1·2·3·4가동,종로1·2·3·4가동사전투표소,NaN,1,"종로구청(3층, 종로가족관)",서울특별시 종로구 삼봉로 43 (수송동),3층,7
7,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,종로5·6가동,종로5·6가동사전투표소,NaN,1,"종로5·6가동주민센터(4층, 다목적회의실)",서울특별시 종로구 종로35가길 19 (효제동),4층,8
8,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,이화동,이화동사전투표소,NaN,1,"이화동주민센터(4층, 다목적실)",서울특별시 종로구 이화장길 33 (이화동),4층,9
9,제21대 국회의원선거,사전투표소,20200415,서울특별시,종로구,혜화동,혜화동사전투표소,NaN,1,"혜화동주민센터(2층, 혜화홀)",서울특별시 종로구 혜화로 12 (혜화동),2층,10



💾 저장 완료: 03_투표소정보.csv  (82795행 × 13열)
